In [ ]:
# ============================================
# PNEUMOSCAN — Stage-2 (Bacterial vs Viral) Evaluator with UI Button
# Uses your saved Stage-2 model and your test split in Drive
# ============================================

# ---- Imports ----
import os, numpy as np, matplotlib.pyplot as plt
from itertools import product
from IPython.display import display, clear_output
import ipywidgets as widgets

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import load_model

from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_fscore_support
)

# ---- 1) Mount Drive ----
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print("Drive mount note:", e)

# ---- 2) Fixed Paths (Stage-2) ----
MODEL_PATH = "/content/drive/MyDrive/pneumonia_detection_system/model/New_efficientnetb0_stage2_bac_vs_vir.keras"
TEST_DIR   = "/content/drive/MyDrive/Pneumonia_Stage2_Dataset/test"   # contains Bacterial/Viral folders

IMG_SIZE   = (224, 224)
BATCH_SIZE = 16
CLASSES    = ['Bacterial Pneumonia', 'Viral Pneumonia']  # IMPORTANT: must match folder names

# ---- UI Elements ----
hdr = widgets.HTML("<h3> Pneumonia_Detection_System — Stage-2 Model Evaluator (Bacterial vs Viral)</h3>")
status_box = widgets.HTML("<b>Status:</b> Initializing...")
evaluate_btn = widgets.Button(description="Evaluate", button_style='primary', tooltip="Run evaluation on the Stage-2 test set")
evaluate_btn.layout.width = "200px"
box = widgets.VBox([hdr, status_box, evaluate_btn])
display(box)

# Globals to hold objects across button click
_model = None
_test_gen = None
_class_indices = None

# ---- 3) Load Model ----
if not os.path.exists(MODEL_PATH):
    status_box.value = f"<b>Status:</b>  Model not found at: <code>{MODEL_PATH}</code>"
    raise SystemExit

status_box.value = "<b>Status:</b> Loading model…"
_model = load_model(MODEL_PATH)
status_box.value = "<b>Status:</b>  Model loaded."

# ---- 4) Load Test Dataset ----
if not os.path.isdir(TEST_DIR):
    status_box.value = f"<b>Status:</b>  Test folder not found at: <code>{TEST_DIR}</code>"
    raise SystemExit

status_box.value = "<b>Status:</b> Loading Stage-2 test set…"

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
_test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
    classes=CLASSES
)

_class_indices = _test_gen.class_indices  # {'Bacterial Pneumonia':0,'Viral Pneumonia':1}
status_box.value = (f"<b>Status:</b>  Test set ready "
                    f"(Bacterial={(_test_gen.classes==_class_indices['Bacterial Pneumonia']).sum()}, "
                    f"Viral={(_test_gen.classes==_class_indices['Viral Pneumonia']).sum()}). "
                    f"<br> Click <b>Evaluate</b> to compute metrics.")

# ---- 5) Helper: Pretty Confusion Matrix Plot (no seaborn) ----
def plot_confusion_matrix(cm, class_names):
    fig = plt.figure(figsize=(5, 4), dpi=120)
    plt.imshow(cm, interpolation='nearest')
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45)
    plt.yticks(tick_marks, class_names)

    thresh = cm.max() / 2.0 if cm.size else 0.5
    for i, j in product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    return fig

# ---- 6) Helper: Plot ROC Curve for a specific class ----
def plot_roc_curve(y_true, y_prob, class_idx, class_name):
    """Plot ROC curve for a specific class (one-vs-rest)"""
    y_true_binary = (y_true == class_idx).astype(int)
    class_scores = y_prob[:, class_idx]

    try:
        auc_score = roc_auc_score(y_true_binary, class_scores)
        fpr, tpr, _ = roc_curve(y_true_binary, class_scores)

        fig = plt.figure(figsize=(5, 4), dpi=120)
        plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc_score:.4f})', linewidth=2)
        plt.plot([0, 1], [0, 1], linestyle='--', linewidth=1, color='gray')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate (Sensitivity)')
        plt.title(f'ROC Curve — {class_name}')
        plt.legend(loc="lower right")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        return fig, auc_score
    except Exception as e:
        print(f"ROC curve for {class_name} skipped: {e}")
        return None, float('nan')

# ---- 7) Evaluation on Button Click ----
out_area = widgets.Output()
display(out_area)

def on_evaluate_clicked(b):
    with out_area:
        clear_output(wait=True)
        if _model is None or _test_gen is None:
            print("Model/Test generator not initialized.")
            return

        print(" Running evaluation …")

        # a) Evaluate() for Loss/Accuracy
        loss, acc = _model.evaluate(_test_gen, verbose=0)
        print(f"\n=== Overall Metrics ===")
        print(f"Test Loss     : {loss:.5f}")
        print(f"Test Accuracy : {acc*100:.2f}%")

        # b) Predictions & Confusion Matrix
        y_true = _test_gen.classes
        y_prob = _model.predict(_test_gen, verbose=0)
        y_pred = np.argmax(y_prob, axis=1)

        # Ensure label order [Bacterial, Viral]
        bacterial_idx = _class_indices['Bacterial Pneumonia']
        viral_idx     = _class_indices['Viral Pneumonia']
        cm = confusion_matrix(y_true, y_pred, labels=[bacterial_idx, viral_idx])

        # cm layout (labels=[Bacterial, Viral]):
        # [[TN, FP],
        #  [FN, TP]]
        TN, FP, FN, TP = cm.ravel()

        # c) Derived Metrics (Positive class = Viral Pneumonia)
        eps = 1e-12
        sensitivity = TP / (TP + FN + eps)  # Recall for Viral
        specificity = TN / (TN + FP + eps)  # True Negative rate wrt Bacterial
        precision_pos = TP / (TP + FP + eps)
        recall_pos    = sensitivity
        f1_pos        = 2 * precision_pos * recall_pos / (precision_pos + recall_pos + eps)

        # Macro P/R/F1 for completeness
        prfs = precision_recall_fscore_support(
            y_true, y_pred, labels=[bacterial_idx, viral_idx], zero_division=0
        )
        precision_macro = prfs[0].mean()
        recall_macro    = prfs[1].mean()
        f1_macro        = prfs[2].mean()

        # d) AUC-ROC for BOTH classes (NEW IMPROVEMENT)
        # AUC for Viral (as positive class)
        y_true_viral = (y_true == viral_idx).astype(int)
        viral_scores = y_prob[:, viral_idx]
        try:
            auc_viral = roc_auc_score(y_true_viral, viral_scores)
        except ValueError:
            auc_viral = float('nan')

        # AUC for Bacterial (as positive class) - NEW ADDITION
        y_true_bacterial = (y_true == bacterial_idx).astype(int)
        bacterial_scores = y_prob[:, bacterial_idx]
        try:
            auc_bacterial = roc_auc_score(y_true_bacterial, bacterial_scores)
        except ValueError:
            auc_bacterial = float('nan')

        # Average AUC (Macro-average) - NEW ADDITION
        if not np.isnan(auc_viral) and not np.isnan(auc_bacterial):
            auc_macro = (auc_viral + auc_bacterial) / 2
        else:
            auc_macro = float('nan')

        # e) Classification Report
        print("\n=== Classification Report (per class) ===")
        print(classification_report(
            y_true, y_pred,
            target_names=['Bacterial Pneumonia', 'Viral Pneumonia'],
            labels=[bacterial_idx, viral_idx],
            digits=4
        ))

        # f) Confusion Matrix (counts + TP/TN/FP/FN)
        print("=== Confusion Matrix (counts) ===")
        print(cm)
        print(f"\nTP  : {TP} ")
        print(f"TN : {TN} ")
        print(f"FP : {FP} ")
        print(f"FN  : {FN} ")

        # g) Key Metrics Summary (BOTH classes separately) - IMPROVED
        print("\n" + "="*50)
        print("KEY METRICS SUMMARY")
        print("="*50)

        print("\n--- For VIRAL PNEUMONIA (as Positive Class) ---")
        print(f"Precision    : {precision_pos:.4f}")
        print(f"Recall/Sensitivity : {sensitivity:.4f}")
        print(f"F1-Score     : {f1_pos:.4f}")
        print(f"AUC-ROC      : {auc_viral:.4f}")

        print("\n--- For BACTERIAL PNEUMONIA (as Positive Class) ---")
        # Calculate metrics for Bacterial as positive
        precision_bacterial = TN / (TN + FN + eps)  # Precision when Bacterial is positive
        recall_bacterial = TN / (TN + FP + eps)     # Recall/Sensitivity when Bacterial is positive
        f1_bacterial = 2 * precision_bacterial * recall_bacterial / (precision_bacterial + recall_bacterial + eps)
        print(f"Precision    : {precision_bacterial:.4f}")
        print(f"Recall/Sensitivity : {recall_bacterial:.4f}")
        print(f"F1-Score     : {f1_bacterial:.4f}")
        print(f"AUC-ROC      : {auc_bacterial:.4f}")

        print("\n--- General Metrics ---")
        print(f"Specificity (for Bacterial) : {specificity:.4f}")
        print(f"Macro-average AUC           : {auc_macro:.4f}")

        print("\n=== Macro Averages (Bacterial & Viral) ===")
        print(f"Precision (macro): {precision_macro:.4f}")
        print(f"Recall (macro)   : {recall_macro:.4f}")
        print(f"F1 (macro)       : {f1_macro:.4f}")

        # h) Visuals — Confusion Matrix Heatmap
        fig_cm = plot_confusion_matrix(cm, ['Bacterial Pneumonia', 'Viral Pneumonia'])
        plt.show(fig_cm)

        # i) ROC Curves for BOTH classes (IMPROVED - Now shows both)
        print("\n" + "="*50)
        print("ROC CURVES (One-vs-Rest)")
        print("="*50)

        # Create subplot for both ROC curves
        fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=120)

        # ROC Curve for Bacterial
        y_true_bacterial = (y_true == bacterial_idx).astype(int)
        bacterial_scores = y_prob[:, bacterial_idx]
        try:
            fpr_bact, tpr_bact, _ = roc_curve(y_true_bacterial, bacterial_scores)
            auc_bact = auc_bacterial
            axes[0].plot(fpr_bact, tpr_bact, label=f'ROC curve (AUC = {auc_bact:.4f})', linewidth=2, color='blue')
            axes[0].plot([0, 1], [0, 1], linestyle='--', linewidth=1, color='gray')
            axes[0].set_xlim([0.0, 1.0])
            axes[0].set_ylim([0.0, 1.05])
            axes[0].set_xlabel('False Positive Rate')
            axes[0].set_ylabel('True Positive Rate (Sensitivity)')
            axes[0].set_title('ROC Curve — Bacterial Pneumonia\n(As Positive Class)')
            axes[0].legend(loc="lower right")
            axes[0].grid(True, alpha=0.3)
        except Exception as e:
            axes[0].text(0.5, 0.5, f"ROC unavailable\n{e}",
                        ha='center', va='center', transform=axes[0].transAxes)
            axes[0].set_title('ROC Curve — Bacterial Pneumonia')

        # ROC Curve for Viral
        y_true_viral = (y_true == viral_idx).astype(int)
        viral_scores = y_prob[:, viral_idx]
        try:
            fpr_viral, tpr_viral, _ = roc_curve(y_true_viral, viral_scores)
            auc_vir = auc_viral
            axes[1].plot(fpr_viral, tpr_viral, label=f'ROC curve (AUC = {auc_vir:.4f})', linewidth=2, color='red')
            axes[1].plot([0, 1], [0, 1], linestyle='--', linewidth=1, color='gray')
            axes[1].set_xlim([0.0, 1.0])
            axes[1].set_ylim([0.0, 1.05])
            axes[1].set_xlabel('False Positive Rate')
            axes[1].set_ylabel('True Positive Rate (Sensitivity)')
            axes[1].set_title('ROC Curve — Viral Pneumonia\n(As Positive Class)')
            axes[1].legend(loc="lower right")
            axes[1].grid(True, alpha=0.3)
        except Exception as e:
            axes[1].text(0.5, 0.5, f"ROC unavailable\n{e}",
                        ha='center', va='center', transform=axes[1].transAxes)
            axes[1].set_title('ROC Curve — Viral Pneumonia')

        plt.tight_layout()
        plt.show()

        # Additional single ROC curve for comparison (original style)
        print("\n--- Combined ROC Analysis ---")
        fig_single = plt.figure(figsize=(5, 4), dpi=120)
        try:
            # Plot both curves on same graph for comparison
            fpr_bact, tpr_bact, _ = roc_curve(y_true_bacterial, bacterial_scores)
            fpr_viral, tpr_viral, _ = roc_curve(y_true_viral, viral_scores)

            plt.plot(fpr_bact, tpr_bact, label=f'Bacterial (AUC = {auc_bacterial:.4f})', linewidth=2, color='blue')
            plt.plot(fpr_viral, tpr_viral, label=f'Viral (AUC = {auc_viral:.4f})', linewidth=2, color='red')
            plt.plot([0, 1], [0, 1], linestyle='--', linewidth=1, color='gray')
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate (Sensitivity)')
            plt.title('ROC Curves Comparison\n(One-vs-Rest)')
            plt.legend(loc="lower right")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f"Combined ROC curve skipped: {e}")

        print("\n" + "="*50)
        print("EVALUATION COMPLETE")
        print("="*50)
        print(f"\nTotal Samples Evaluated: {len(y_true)}")
        print(f"  • Bacterial Pneumonia: {sum(y_true == bacterial_idx)}")
        print(f"  • Viral Pneumonia: {sum(y_true == viral_idx)}")

        # Final accuracy check
        correct_predictions = (y_true == y_pred).sum()
        calculated_accuracy = correct_predictions / len(y_true)
        print(f"\nManual Accuracy Check: {correct_predictions}/{len(y_true)} = {calculated_accuracy*100:.2f}%")
        print(f"Model.evaluate() Accuracy: {acc*100:.2f}%")

# Only enable button after dataset load
def enable_button_after_load():
    evaluate_btn.disabled = False
    evaluate_btn.button_style = 'success'

evaluate_btn.on_click(on_evaluate_clicked)
enable_button_after_load()

Mounted at /content/drive


Found 200 images belonging to 2 classes.


Output()